# A100 GPU Monitoring Dashboard

Real-time monitoring cho NVIDIA A100 GPU trong Docker JupyterLab.

**Setup:**
1. Upload file này vào JupyterLab trên A100 server
2. Chạy Cell 1 để install dependencies
3. Chạy Cell 3 để bắt đầu monitoring

**Access:** https://grew-hypothesis-mothers-flooring.trycloudflare.com

In [ ]:
# Cell 1: Install dependencies
!pip install gpustat nvitop pandas matplotlib

In [ ]:
# Cell 2: GPU monitoring functions
import subprocess
import json
import time
from datetime import datetime

def get_gpu_stats():
    """Get current GPU statistics from nvidia-smi"""
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw,power.limit', 
         '--format=csv,noheader,nounits'], 
        capture_output=True, 
        text=True
    )
    
    gpu_util, mem_used, mem_total, temp, power_draw, power_limit = result.stdout.strip().split(',')
    
    return {
        'timestamp': datetime.now().isoformat(),
        'gpu_utilization': float(gpu_util),
        'memory_used_mb': float(mem_used),
        'memory_total_mb': float(mem_total),
        'memory_percent': (float(mem_used) / float(mem_total)) * 100,
        'temperature_c': float(temp),
        'power_draw_w': float(power_draw),
        'power_limit_w': float(power_limit),
        'power_percent': (float(power_draw) / float(power_limit)) * 100
    }

def check_gpu_health(stats):
    """Check if GPU metrics are within healthy ranges"""
    warnings = []
    
    if stats['temperature_c'] > 80:
        warnings.append(f"⚠️ High temperature: {stats['temperature_c']}°C")
    
    if stats['memory_percent'] > 90:
        warnings.append(f"⚠️ High memory usage: {stats['memory_percent']:.1f}%")
    
    if stats['power_percent'] > 95:
        warnings.append(f"⚠️ High power draw: {stats['power_percent']:.1f}%")
    
    return warnings

print("✅ GPU monitoring functions loaded")

In [ ]:
# Cell 3: Real-time monitoring dashboard
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

stats_history = []
max_history = 100  # Keep last 100 data points

print("🚀 Starting GPU monitoring... (Press Interrupt to stop)\n")

try:
    while True:
        # Get current stats
        stat = get_gpu_stats()
        stats_history.append(stat)
        
        # Keep only recent history
        if len(stats_history) > max_history:
            stats_history.pop(0)
        
        # Check health
        health_warnings = check_gpu_health(stat)
        
        # Clear output and plot
        clear_output(wait=True)
        
        # Create dashboard
        df = pd.DataFrame(stats_history)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'A100 GPU Monitoring Dashboard - {stat["timestamp"]}', fontsize=16, fontweight='bold')
        
        # GPU Utilization
        axes[0,0].plot(df.index, df['gpu_utilization'], color='#2ecc71', linewidth=2)
        axes[0,0].fill_between(df.index, df['gpu_utilization'], alpha=0.3, color='#2ecc71')
        axes[0,0].set_title('GPU Utilization (%)', fontsize=12, fontweight='bold')
        axes[0,0].set_ylim(0, 100)
        axes[0,0].grid(True, alpha=0.3)
        axes[0,0].axhline(y=70, color='orange', linestyle='--', alpha=0.5, label='Target: 70%')
        axes[0,0].legend()
        
        # Memory Usage
        axes[0,1].plot(df.index, df['memory_percent'], color='#3498db', linewidth=2)
        axes[0,1].fill_between(df.index, df['memory_percent'], alpha=0.3, color='#3498db')
        axes[0,1].set_title(f'Memory Usage (%) - {stat["memory_used_mb"]:.0f}/{stat["memory_total_mb"]:.0f} MB', 
                           fontsize=12, fontweight='bold')
        axes[0,1].set_ylim(0, 100)
        axes[0,1].grid(True, alpha=0.3)
        axes[0,1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='Warning: 90%')
        axes[0,1].legend()
        
        # Temperature
        axes[1,0].plot(df.index, df['temperature_c'], color='#e74c3c', linewidth=2)
        axes[1,0].fill_between(df.index, df['temperature_c'], alpha=0.3, color='#e74c3c')
        axes[1,0].set_title(f'Temperature (°C) - Current: {stat["temperature_c"]:.1f}°C', 
                           fontsize=12, fontweight='bold')
        axes[1,0].set_ylim(0, 100)
        axes[1,0].grid(True, alpha=0.3)
        axes[1,0].axhline(y=80, color='orange', linestyle='--', alpha=0.5, label='Warning: 80°C')
        axes[1,0].legend()
        
        # Power Draw
        axes[1,1].plot(df.index, df['power_percent'], color='#9b59b6', linewidth=2)
        axes[1,1].fill_between(df.index, df['power_percent'], alpha=0.3, color='#9b59b6')
        axes[1,1].set_title(f'Power Draw (%) - {stat["power_draw_w"]:.0f}/{stat["power_limit_w"]:.0f} W', 
                           fontsize=12, fontweight='bold')
        axes[1,1].set_ylim(0, 100)
        axes[1,1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Display health warnings
        if health_warnings:
            print("\n⚠️ HEALTH WARNINGS:")
            for warning in health_warnings:
                print(f"  {warning}")
        else:
            print("\n✅ All metrics healthy")
        
        # Display current stats
        print(f"\n📊 Current Stats:")
        print(f"  GPU Utilization: {stat['gpu_utilization']:.1f}%")
        print(f"  Memory: {stat['memory_used_mb']:.0f}/{stat['memory_total_mb']:.0f} MB ({stat['memory_percent']:.1f}%)")
        print(f"  Temperature: {stat['temperature_c']:.1f}°C")
        print(f"  Power: {stat['power_draw_w']:.0f}/{stat['power_limit_w']:.0f} W ({stat['power_percent']:.1f}%)")
        
        time.sleep(5)  # Update every 5 seconds
        
except KeyboardInterrupt:
    print("\n\n🛑 Monitoring stopped")
    
    # Save stats to file
    if stats_history:
        df_final = pd.DataFrame(stats_history)
        filename = f"gpu_stats_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df_final.to_csv(filename, index=False)
        print(f"📁 Stats saved to: {filename}")

In [ ]:
# Cell 4: Quick GPU check (single snapshot)
stat = get_gpu_stats()
warnings = check_gpu_health(stat)

print("🖥️ A100 GPU Status Snapshot")
print("=" * 50)
print(f"Timestamp: {stat['timestamp']}")
print(f"GPU Utilization: {stat['gpu_utilization']:.1f}%")
print(f"Memory: {stat['memory_used_mb']:.0f}/{stat['memory_total_mb']:.0f} MB ({stat['memory_percent']:.1f}%)")
print(f"Temperature: {stat['temperature_c']:.1f}°C")
print(f"Power: {stat['power_draw_w']:.0f}/{stat['power_limit_w']:.0f} W ({stat['power_percent']:.1f}%)")
print("=" * 50)

if warnings:
    print("\n⚠️ Warnings:")
    for w in warnings:
        print(f"  {w}")
else:
    print("\n✅ All metrics healthy")